<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/seguridad-informacion/02_deteccion_intrusiones_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Detección de intrusiones con aprendizaje automático

**Seguridad de la Información** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Describir un flujo de red con características estadísticas.
- Entrenar un clasificador supervisado (bosque aleatorio) para detectar DDoS y escaneos.
- Evaluar con matriz de confusión, precisión y exhaustividad.
- Contrastar con detección de anomalías no supervisada (una sola clase).

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

## 1. Datos
Un IDS basado en flujos no inspecciona el contenido: resume cada conversación en estadísticas. Generamos datos sintéticos con tres clases: tráfico **normal**, **DDoS** (inundación SYN) y **escaneo** de puertos.

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(2026)

def generar_flujos(n_normal=3000, n_ddos=800, n_scan=400):
    """Flujos sintéticos inspirados en conjuntos como LATAM-DDoS-IoT o SDNFlow."""
    def bloque(n, dur, pk, by_pk, puertos, syn, etiqueta):
        d = pd.DataFrame({
            "duracion_s": rng.gamma(*dur, n),
            "paquetes": rng.poisson(pk, n) + 1,
            "bytes_por_paquete": rng.normal(*by_pk, n).clip(40, 1500),
            "puertos_destino": rng.poisson(puertos, n) + 1,
            "fraccion_syn": rng.beta(*syn, n),
        })
        d["tasa_pps"] = d["paquetes"] / (d["duracion_s"] + 1e-3)
        d["etiqueta"] = etiqueta
        return d
    return pd.concat([
        bloque(n_normal, (2, 3), 40, (700, 250), 0.3, (1, 12), "normal"),
        bloque(n_ddos, (1, 0.3), 400, (60, 10), 0.2, (12, 2), "ddos"),
        bloque(n_scan, (1, 1), 6, (60, 5), 25, (8, 3), "escaneo"),
    ]).sample(frac=1, random_state=1).reset_index(drop=True)

flujos = generar_flujos()
flujos.head()

In [ ]:
flujos.groupby('etiqueta').median(numeric_only=True).round(2)

**Ejercicio 1.** ¿Cuántos flujos hay en total? → `resp_1`

In [ ]:
resp_1 = None

## 2. Clasificador supervisado

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X = flujos.drop(columns="etiqueta"); y = flujos["etiqueta"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
pred = rf.predict(X_te)
print(classification_report(y_te, pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_te, pred); plt.show()

**Ejercicio 2.** Use `rf.feature_importances_` para identificar la característica **más importante** y guarde su nombre (texto) en `resp_2`. ¿Tiene sentido desde el punto de vista de un ataque DDoS?

In [ ]:
resp_2 = None

## 3. Detección de anomalías (una sola clase)
En la práctica rara vez tenemos ejemplos etiquetados de todos los ataques. Un detector de **una sola clase** aprende solo el tráfico normal y marca lo que se desvía. Ver: P. Arregocés, J. Vergara, S. A. Gutiérrez, J. F. Botero, *Network-based intrusion detection: A one-class classification approach*, NOMS 2022.

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=0).fit(X_tr[y_tr == "normal"])
anom = iso.predict(X_te) == -1
print(pd.crosstab(y_te, anom, colnames=["marcado como anómalo"]))

**Ejercicio 3.** Varíe `contamination` entre 0.01 y 0.2. Grafique la tasa de detección de ataques y la tasa de falsos positivos del tráfico normal. ¿Qué valor escogería para un centro de operaciones con pocos analistas?

In [ ]:
# Experimento del ejercicio 3

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 4200)
verificar('Ejercicio 2', resp_2, 'paquetes')

## Para ir más lejos
Repita el análisis con un conjunto real como **LATAM-DDoS-IoT** (IEEE DataPort) o **SDNFlow** (IEEE Access, 2024), ambos desarrollados con participación del grupo GITA.